In [ ]:
import os
import re
import cv2
import numpy as np
import SimpleITK as sitk
from tqdm import tqdm
import nrrd

# Define the input folder containing JPG images
input_folder = '/home/projects/registration/sriprabha/vinoth/data_file/final_cropped_222_OCT9'

# Function to extract the image number from the filename
def extract_image_number(filename):
    pattern = re.compile(r'_(\d+)_original\.jpg$')
    match = pattern.search(filename)
    if match:
        return int(match.group(1))
    else:
        print(f"Warning: No number found in filename {filename}. Skipping.")
        return None

# List and sort all JPG images in the folder based on the extracted image number
image_files = [f for f in os.listdir(input_folder) if f.endswith(".jpg")]
image_files.sort(key=lambda f: extract_image_number(f) or float('inf'))

# Function to read and extract the blue channel from an image without resizing
def extract_blue_channel(img_path):
    image = cv2.imread(img_path)
    if image is None:
        raise ValueError(f"Unable to read image {img_path}.")
    
    # Extract the blue channel (OpenCV loads images in BGR format)
    blue_channel = image[:, :, 0]
    
    # Ensure the blue channel is of dtype uint8
    blue_channel = blue_channel.astype(np.uint8)
    
    return blue_channel

# Function to align the centroids of the fixed and moving images
def align_centroids(fixed_image, moving_image):
    fixed_center = np.array(fixed_image.shape) / 2.0
    moving_center = np.array(moving_image.shape) / 2.0
    translation = fixed_center - moving_center

    # Create a translation transform
    translation_transform = sitk.TranslationTransform(2, translation.tolist())
    moving_image_sitk = sitk.GetImageFromArray(moving_image.astype(np.float32))

    # Apply the translation
    resampled_image = sitk.Resample(
        moving_image_sitk,
        sitk.GetImageFromArray(fixed_image.astype(np.float32)),
        translation_transform,
        sitk.sitkLinear,
        0.0,
        moving_image_sitk.GetPixelID(),
    )

    # Convert back to a NumPy array
    aligned_image = sitk.GetArrayFromImage(resampled_image)

    return aligned_image

# Registration setup using SimpleITK
def register_images(fixed_image, moving_image):
    registration_method = sitk.ImageRegistrationMethod()
    registration_method.SetMetricAsMeanSquares()
    registration_method.SetInterpolator(sitk.sitkLinear)
    registration_method.SetOptimizerAsGradientDescent(learningRate=1.0, numberOfIterations=200)
    
    # Translation only transform
    initial_transform = sitk.TranslationTransform(2)
    registration_method.SetInitialTransform(initial_transform, inPlace=False)

    # Execute the registration
    final_transform = registration_method.Execute(fixed_image, moving_image)
    resampled_image = sitk.Resample(moving_image, fixed_image, final_transform, sitk.sitkLinear, 0.0, moving_image.GetPixelID())

    return resampled_image

# Read the first image as the fixed image
fixed_image_path = os.path.join(input_folder, image_files[0])
fixed_image_blue = extract_blue_channel(fixed_image_path)

# Convert the fixed image to SimpleITK format and float32
fixed_image_sitk = sitk.GetImageFromArray(fixed_image_blue.astype(np.float32))

# Initialize the list for registered images
registered_images = [fixed_image_blue]

# Register all moving images to the fixed image
for filename in tqdm(image_files[1:], desc="Registering images", unit="image"):
    img_path = os.path.join(input_folder, filename)
    moving_image_blue = extract_blue_channel(img_path)

    # Align the centroids of the moving and fixed images
    moving_image_aligned = align_centroids(fixed_image_blue, moving_image_blue)

    # Convert to SimpleITK image in float32
    moving_image_sitk = sitk.GetImageFromArray(moving_image_aligned.astype(np.float32))

    # Perform registration
    registered_image_sitk = register_images(fixed_image_sitk, moving_image_sitk)

    # Convert back to NumPy array and uint8
    registered_image = sitk.GetArrayFromImage(registered_image_sitk)
    registered_image_uint8 = np.clip(registered_image, 0, 255).astype(np.uint8)

    # Append the registered uint8 image to the list
    registered_images.append(registered_image_uint8)

# Output the final shape of the registered image stack
registered_image_stack = np.stack(registered_images, axis=0)
print(f"Final shape of the registered image stack: {registered_image_stack.shape}")
print(f"Data type of the registered image stack: {registered_image_stack.dtype}")

# Convert the list of registered images to a NumPy array and ensure 8-bit unsigned char format
registered_images_array = np.stack(registered_images, axis=0).astype(np.uint8)

# Save the final registered stack as a .nrrd file
nrrd.write('RR_stack_fid_remov_FULL_222.nrrd', registered_images_array)